# Linear Regression Model From Scratch
Given our response vector $Y$, a predictor matrix $X$, and a coefficients vector $\hat{\beta}$, our model will approxiamte the response as a linear relation:
$$Y \approx X\hat{\beta}.$$
First we'll do this by the **OLS** (ordinary least squares) method by minimizing the **residual sum of squares**
$$RSS=\sum_{i=1}^{n}(y_{i}-\hat{y_{i}})^{2}=\sum_{i=1}^{n}(y_{i}-\hat{\beta_{0}}-\hat{\beta_{1}}x_{i 1}-\dots-\hat{\beta_{p}}x_{ip})^{2},$$
via the **normal equation**:
$$\theta = \hat{\beta} = (X^{T}X)^{-1}X^{T}Y,$$
wich gives us the best estimate for $\hat{\beta}$ given $X$ and $Y$.

In [2]:
# Import numpy, the main library for creating the model
import numpy as np

In [3]:
class LinearRegression:
    """A linear regression model with both a fit() and predict() methods.

    The model is made to work with quantitative predictors, and qualitative predictors 
    should be taken into account when creating the observations matrix.

    Attributes:
        coefficients(np.array(float)): One dimensional array with the model estimated coefficients.
        intercept(float): Intercept coefficient.
        fit_intercept(bool): Indicates if the model will have an intercept.
    """

    def __init__(self, intercept : bool = True):
        """Initializes the model with or without an intercept.

        Args:
            intercept(bool): Indicates if the model will have an intercept. Set to True by default.
        """
        self.coefficients = None
        self.intercept = None
        self.fit_intercept = intercept

    def fit(self, X : np.ndarray, y : np.ndarray) -> LinearRegression:
        """Fits the model given observations.

        Args:
            X(np.array(np.array(float))): Matrix containing the observations predictors.
            y(np.arrray(float)): Response vector for the observations.

        Returns:
            LinearRegression class with fitted coefficients.
        
        Raises:
            ValueError if the matrices are of the wrong sizes.
        """
        # Convert X and y to arrays in case they are not
        X = np.asarray(X)
        y = np.asarray(y)

        # Add bias term
        if self.fit_intercept:
            X = np.column_stack([np.ones(X.shape[0]), X])

        try:
            A : np.ndarray = X.T @ X 
            b : np.ndarray = X.T @ y
            # Solves the system Ax=b
            theta : np.ndarray = np.linalg.solve(A, b)

            if self.fit_intercept:
                self.intercept = theta[0]
                self.coefficients = theta[1:]
            else:
                self.intercept = 0
                self.coefficients = theta
            
            return self
        
        except ValueError:
            print("X must have (n,p) shape, and y must have shape n")
        

    def predict(self, X : np.ndarray) -> float:
        """Gives a prediction given a set of data.

        Args:
            X(np.array(float)): Vector containing the data to make a prediction.

        Returns: 
            Prediction(float) for the vector X.
        """
        # Checks if the model is fitted
        if self.coefficients is None:
            raise ValueError("Model not fitted. Call fit() first.")

        X : np.ndarray = np.asarray(X)

        return X @ self.coefficients + self.intercept


In [4]:
# Synhetic observations
n : int = 4000   # number of observations
p : int = 10     # number of predictors
error : np.ndarray = np.random.normal(loc=0, scale=1, size=n)         # (nx1) error vector
X : np.ndarray = np.random.uniform(size = (n,p))                      # (nxp) observations
beta0 : float = 5                                                   # intercept
beta : np.ndarray = np.random.randint(low=3, high=20, size=p)         # (px1) random predictors
Y : np.ndarray = beta0 + (X @ beta) + error

In [5]:
beta

array([ 8, 17,  9, 19,  3, 11, 15,  6, 19, 16])

In [6]:
model : LinearRegression = LinearRegression()

In [7]:
results : LinearRegression = model.fit(X, Y)
print(f"Los coeficientes son: {results.intercept}, {results.coefficients}")

Los coeficientes son: 5.178885280451499, [ 7.86701631 16.96963117  9.04107592 19.00896594  2.93903051 10.98406486
 15.00759998  5.89706334 19.00690318 15.92270945]


In [8]:
X_predict : np.ndarray = np.random.normal(loc=0, scale=1, size=p)
X_predict

array([-0.66709688, -0.01425196,  0.09326582,  0.75651933,  0.76417425,
        0.03771358, -0.7108461 ,  0.58983802,  0.14646042, -0.260822  ])

In [9]:
results.predict(X_predict)

np.float64(9.014010379503922)

Now, let's compare the model's estimates for $\beta$ to scikit-learn's linear regression model.

In [17]:
# Import sklearn's linear regression model
from sklearn.linear_model import LinearRegression as LR
reg = LR().fit(X,Y)
reg.coef_

array([ 7.86701631, 16.96963117,  9.04107592, 19.00896594,  2.93903051,
       10.98406486, 15.00759998,  5.89706334, 19.00690318, 15.92270945])

In [16]:
for a, b in zip(reg.coef_, results.coefficients):
    print(a-b)

-1.412203687323199e-13
-1.5631940186722204e-13
1.829647544582258e-13
-4.369837824924616e-13
4.3876013933186186e-13
5.879741138414829e-13
3.3573144264664734e-13
1.900701818158268e-13
5.080380560684716e-13
-1.1191048088221578e-13


We can see that the coefficients estimated by sklearn's LinearRegression and the coefficients estimated by our own linear model match up to 13 decimal places.